# 04 · Intermediate · Restricted access and organization views

Not every organization lets users read the `system` catalog. Some publish their own
views instead. The `sources:` section of the policy points the scanner at whatever
you can read.

| Your situation | `sources:` setting |
|---|---|
| You can read `system.information_schema` | nothing (default) |
| `system` is blocked | `information_schema: per_catalog` |
| Your org has a schema of metadata views | `information_schema: governance.uc_meta` |
| One view is different or renames columns | `tables: {table: ..., columns: {...}}` |

In [ ]:
%pip install /Volumes/main/default/lakehouse_gc/lakehouse_gc-0.2.2-py3-none-any.whl

In [ ]:
from lakehouse_gc import check_access, run
from lakehouse_gc.core import loads_policy

BASE = """
policy_version: 1
scope:
  include_catalogs: ["main"]
"""

# 1. Default: system.information_schema. If this says "cannot run", try the next cells.
default_policy = loads_policy(BASE)
print(check_access(spark, policy=default_policy, scanners=["uc"]).summary())

In [ ]:
# 2. The system catalog is blocked: read each catalog's own information_schema instead.
#    Catalogs are listed with SHOW CATALOGS.
per_catalog = loads_policy(BASE + "sources:\n  information_schema: per_catalog\n")
print(check_access(spark, policy=per_catalog, scanners=["uc"]).summary())

result = run(policy=per_catalog, scanners=["uc"], spark=spark)
print(result.summary())

In [ ]:
# 3. Your organization publishes views. Point at a schema of views with the same shape, or
#    repoint one view and map renamed columns (logical name: your column name).
#    check_access shows each source as available, partial (some columns missing, so only
#    the checks that need them are unknown), or unavailable, with how to fix it.
org_views = loads_policy(
    BASE
    + """
sources:
  information_schema: per_catalog
  tables:
    table: governance.uc_meta.all_tables
    columns:
      table_owner: owner_principal
      last_altered: changed_at
"""
)
report = check_access(spark, policy=org_views, scanners=["uc"])
display(report.to_dataframe(spark))
print("uc can run:", report.can_run("uc"))